In [3]:
import requests as req
import os
from dotenv import load_dotenv
import json

In [4]:

load_dotenv()
base_url = os.getenv("BASE_URL")
api_key = os.getenv("YOUTUBE_API_KEY")

channel_handle = os.getenv("CHANNEL_HANDLE")

In [5]:
params = {
    "part": "brandingSettings",
    "forHandle": channel_handle
    }
res = req.get(f"{base_url}/channels",params=params | {"key": api_key}).json()["items"][0]["brandingSettings"]["image"]["bannerExternalUrl"]


In [6]:
res

'https://yt3.googleusercontent.com/8sT9BbnbNORM5aSzSyHDFV_1ItIwaiLq4ctD4aLeMJnLiZYNCr73eN_-0Xrhy5joVL_Ewet5'

In [16]:
import json
import pandas as pd

In [17]:
save_folder_path = "../data/processed/"

df = pd.read_json('../data/raw/youtube_raw_data.json')
df

,videoId,title,publishedAt,duration,viewCount,likeCount,commentCount,thumbnailUrl,topicCategories
0,z7Qp9Q_Br88,Laughing Gas Is Not So Funny,2026-04-27T14:01:38Z,PT1M15S,415167,23850,666,https://i.ytimg.com/vi/z7Qp9Q_Br88/mqdefault.jpg,[https://en.wikipedia.org/wiki/Entertainment]
1,FZbMPtLRr00,Fast-Forward Your Life,2026-04-23T14:00:23Z,PT1M5S,524885,20570,843,https://i.ytimg.com/vi/FZbMPtLRr00/mqdefault.jpg,[]
2,dl0-TveDDGA,Our Minds Are Weirder than You Think,2026-04-21T14:00:01Z,PT12M22S,1625517,82299,3146,https://i.ytimg.com/vi/dl0-TveDDGA/mqdefault.jpg,[https://en.wikipedia.org/wiki/Knowledge]
3,VYhUa3WC4nU,Why Being Poor Is Expensive,2026-04-16T14:00:55Z,PT1M13S,1574989,99860,3026,https://i.ytimg.com/vi/VYhUa3WC4nU/mqdefault.jpg,[]
4,lRvN5SJR2LQ,How Caffeine Works,2026-04-13T14:01:13Z,PT1M12S,803163,39435,828,https://i.ytimg.com/vi/lRvN5SJR2LQ/mqdefault.jpg,"[https://en.wikipedia.org/wiki/Health, https:/..."
...,...,...,...,...,...,...,...,...,...
352,F3QpgXBtDeo,How The Stock Exchange Works (For Dummies),2013-11-28T17:03:32Z,PT3M34S,8547807,132404,6808,https://i.ytimg.com/vi/F3QpgXBtDeo/mqdefault.jpg,"[https://en.wikipedia.org/wiki/Business, https..."
353,UuGrBhK2c7U,The Gulf Stream Explained,2013-10-11T19:11:39Z,PT5M4S,6328059,66572,2008,https://i.ytimg.com/vi/UuGrBhK2c7U/mqdefault.jpg,[https://en.wikipedia.org/wiki/Society]
354,Uti2niW2BRA,Fracking explained: opportunity or danger,2013-09-03T09:12:24Z,PT5M3S,7461739,102428,8088,https://i.ytimg.com/vi/Uti2niW2BRA/mqdefault.jpg,[https://en.wikipedia.org/wiki/Society]
355,KsF_hdjWJjo,The Solar System -- our home in space,2013-08-22T13:24:56Z,PT7M21S,6581155,85080,6152,https://i.ytimg.com/vi/KsF_hdjWJjo/mqdefault.jpg,[https://en.wikipedia.org/wiki/Knowledge]


In [18]:
# 2. Clean the topic URLs to extract just the category name (e.g., "Entertainment")
df["topicCategories"] = df["topicCategories"].apply(lambda x: [topic.replace("https://en.wikipedia.org/wiki/", "") for topic in x])
df["topicCategories"]

0          [Entertainment]
1                       []
2              [Knowledge]
3                       []
4      [Health, Knowledge]
              ...         
352    [Business, Society]
353              [Society]
354              [Society]
355            [Knowledge]
356            [Knowledge]
Name: topicCategories, Length: 357, dtype: object

In [19]:
# 3. Create the Bridge Table using explode

topic_df = df[["videoId","topicCategories"]].explode("topicCategories").dropna(subset=["topicCategories"]).rename(columns={'topicCategories': 'topic'})
topic_df

,videoId,topic
0,z7Qp9Q_Br88,Entertainment
2,dl0-TveDDGA,Knowledge
4,lRvN5SJR2LQ,Health
4,lRvN5SJR2LQ,Knowledge
5,onFj_RD75nA,Health
...,...,...
352,F3QpgXBtDeo,Society
353,UuGrBhK2c7U,Society
354,Uti2niW2BRA,Society
355,KsF_hdjWJjo,Knowledge


In [20]:
df = df.drop(columns=["topicCategories"])

In [21]:
df.columns

Index(['videoId', 'title', 'publishedAt', 'duration', 'viewCount', 'likeCount',
       'commentCount', 'thumbnailUrl'],
      dtype='object')

In [22]:
df['publishedAt'] = pd.to_datetime(df['publishedAt'])
df['publish_date'] = df['publishedAt'].dt.date
df['publish_hour'] = df['publishedAt'].dt.hour
df['duration'] = pd.to_timedelta(df['duration'])
df['duration_seconds'] = df['duration'].dt.total_seconds().astype(int)

In [23]:
df[['publishedAt', 'publish_date','duration', 'duration_seconds']].head()

,publishedAt,publish_date,duration,duration_seconds
0,2026-04-27 14:01:38+00:00,2026-04-27,0 days 00:01:15,75
1,2026-04-23 14:00:23+00:00,2026-04-23,0 days 00:01:05,65
2,2026-04-21 14:00:01+00:00,2026-04-21,0 days 00:12:22,742
3,2026-04-16 14:00:55+00:00,2026-04-16,0 days 00:01:13,73
4,2026-04-13 14:01:13+00:00,2026-04-13,0 days 00:01:12,72


In [24]:
df.columns

Index(['videoId', 'title', 'publishedAt', 'duration', 'viewCount', 'likeCount',
       'commentCount', 'thumbnailUrl', 'publish_date', 'publish_hour',
       'duration_seconds'],
      dtype='object')

In [ ]:
df.to_csv(f"{save_folder_path}videos.csv", index=False)
topic_df.to_csv(f"{save_folder_path}topics.csv", index=False)